<div style="background:linear-gradient(135deg,#51247a 0%,#7a3fa0 100%);padding:22px 26px;border-radius:10px;margin-bottom:1.2rem;border-bottom:4px solid #f0a500;">
<div style="font-size:2.2rem;margin-bottom:6px;">🏷️</div>
<div style="color:white;font-size:1.5rem;font-weight:700;">Part-of-Speech Tagger</div>
<div style="color:rgba(255,255,255,.8);font-size:.92rem;margin-top:4px;">Tokenise, lemmatise, and tag texts in 65+ languages<br><a href="https://ladal.edu.au/postag.html" style="color:#f0c060;font-size:.82rem;">&#x2192; See the full tutorial</a></div>
</div>


**What this tool does:** Assigns part-of-speech tags, lemmas, and dependency relations to every word using the UDPipe toolkit. Supports 65+ languages.

**Steps:** Upload files &#x2192; Select language &#x2192; Run Analysis &#x2192; Download


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 1 &mdash; Set up the environment</b></div>


In [ ]:
suppressPackageStartupMessages(library(IRdisplay))
IRdisplay::display_html(paste0(
  "<style>",
  "  .jp-InputArea-editor { display: none !important; }",
  "  .jp-CodeCell .jp-Cell-inputWrapper { display: none !important; }",
  "  .show-code .jp-CodeCell .jp-Cell-inputWrapper { display: flex !important; }",
  "  .ladal-toolbar {",
  "    position: sticky; top: 0; z-index: 999;",
  "    background: #51247a; color: white;",
  "    padding: 8px 18px; border-radius: 0 0 8px 8px;",
  "    display: flex; align-items: center; gap: 14px;",
  "    font-family: sans-serif; font-size: .85rem;",
  "    box-shadow: 0 2px 8px rgba(81,36,122,.25);",
  "  }",
  "  .ladal-toolbar b { font-size: 1rem; }",
  "  .ladal-toolbar button {",
  "    background: rgba(255,255,255,.18);",
  "    border: 1px solid rgba(255,255,255,.4);",
  "    color: white; padding: 4px 12px; border-radius: 4px;",
  "    cursor: pointer; font-size: .8rem; font-weight: 600;",
  "  }",
  "  .ladal-toolbar button:hover { background: rgba(255,255,255,.32); }",
  "</style>",
  "<div class=\"ladal-toolbar\">",
  "  <b>&#x1F4D3; LADAL Interactive Notebook</b>",
  "  <button onclick=\"document.body.classList.toggle(&apos;show-code&apos;);",
  "    this.textContent=document.body.classList.contains(&apos;show-code&apos;)",
  "      ?&apos;Hide Code&apos;:&apos;Show Code&apos;\">Show Code</button>",
  "  <span style=\"opacity:.7;font-size:.78rem;\">",
  "    Code is hidden by default &mdash; click to toggle",
  "  </span>",
  "</div>"
))


In [ ]:
# ── Suppress startup messages & load display helpers ────────────────
suppressPackageStartupMessages(library(IRdisplay))
suppressPackageStartupMessages(library(IRkernel))

# ── Colour-coded display helpers ────────────────────────────────────
.box <- function(msg, bg, border, icon="") {
  IRdisplay::display_html(paste0(
    "<div style=\"background:", bg, ";border-left:4px solid ", border,
    ";border-radius:6px;padding:11px 16px;margin:.6rem 0;",
    "font-family:sans-serif;font-size:.9rem;\">",
    if (nzchar(icon)) paste0("<b>", icon, "</b> ") else "",
    msg, "</div>"))
}
ok   <- function(msg) .box(msg, "#eafaf1", "#27ae60", "&#x2705;")
warn <- function(msg) .box(msg, "#fff8e1", "#f0a500", "&#x26A0;&#xFE0F;")
err  <- function(msg) .box(msg, "#fff0f0", "#e74c3c", "&#x274C;")
info <- function(msg) .box(msg, "#f4f0f8", "#51247a", "&#x2139;&#xFE0F;")

# ── Progress bar ─────────────────────────────────────────────────────
.prog <- function(label, value, max=100) {
  pct <- round(100 * value / max)
  IRdisplay::display_html(paste0(
    "<div style=\"font-family:sans-serif;font-size:.85rem;margin:.4rem 0;\">",
    "<span style=\"color:#51247a;font-weight:600;\">", label, "</span><br>",
    "<div style=\"background:#e8e4f0;border-radius:4px;height:10px;margin-top:4px;\">",
    "<div style=\"background:#51247a;width:", pct, "%;height:10px;",
    "border-radius:4px;transition:width .3s;\"></div></div>",
    "<span style=\"color:#888;font-size:.78rem;\">", pct, "%</span></div>"))
}

# ── Upload instructions ──────────────────────────────────────────────
upload_instructions <- function(folder="MyTexts", filetype="txt") {
  IRdisplay::display_html(paste0(
    "<div style=\"background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;",
    "padding:18px 22px;margin:.8rem 0;font-family:sans-serif;\">",
    "<b style=\"color:#51247a;font-size:1rem;\">&#x1F4C2; Upload your files</b><br><br>",
    "<ol style=\"margin:0;padding-left:1.4em;font-size:.9rem;line-height:1.8;\">",
    "<li>Find the <b>file browser panel</b> on the left side of the screen.</li>",
    "<li>Double-click the <b><code>", folder, "</code></b> folder to open it.</li>",
    "<li><b>Drag and drop</b> your <code>.", filetype, "</code> files into that folder.</li>",
    "<li>Come back here and click <b>Run Analysis</b> below.</li>",
    "</ol>",
    "<p style=\"margin:.6rem 0 0 0;font-size:.82rem;color:#888;\">",
    "Only <code>.", filetype, "</code> files are accepted. ",
    "You can upload multiple files at once.</p></div>"))
}

# ── Load plain-text files ────────────────────────────────────────────
load_texts <- function(folder = "notebooks/MyTexts") {
  files <- list.files(folder, pattern = "\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop("No .txt files found in ", folder,
         ". Please upload files into the ", basename(folder), " folder.")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# ── Save texts ───────────────────────────────────────────────────────
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# ── Save Excel ───────────────────────────────────────────────────────
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

ok("Environment ready. Scroll down to configure and run your analysis.")


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 2 &#x1F4C2; &mdash; Upload your texts</b></div>


In [ ]:
upload_instructions("MyTexts", "txt")

<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 3 &mdash; Select language and run</b></div>


In [ ]:
suppressPackageStartupMessages(library(ipywidgets))

lang_choices <- list(
  "English (EWT)"       = "english-ewt",
  "English (GUM)"       = "english-gum",
  "German (GSD)"        = "german-gsd",
  "German (HDT)"        = "german-hdt",
  "French (GSD)"        = "french-gsd",
  "Spanish (AnCora)"    = "spanish-ancora",
  "Spanish (GSD)"       = "spanish-gsd",
  "Italian (ISDT)"      = "italian-isdt",
  "Dutch (Alpino)"      = "dutch-alpino",
  "Portuguese (Bosque)" = "portuguese-bosque",
  "Russian (GSD)"       = "russian-gsd",
  "Chinese (GSD)"       = "chinese-gsd",
  "Arabic (PADT)"       = "arabic-padt",
  "Japanese (GSD)"      = "japanese-gsd"
)
w_lang <- ipywidgets::Dropdown(options=lang_choices, value="english-ewt",
            description="Language:", style=list(description_width="120px"))
w_btn  <- ipywidgets::Button(description="  Run Analysis",
            button_style="primary", icon="tag")
w_out  <- ipywidgets::Output()

ipywidgets::display(ipywidgets::VBox(list(
  ipywidgets::HTML("<b style=\"color:#51247a\">Select language model and run:</b>"),
  w_lang, w_btn, w_out
)))

ipywidgets::observe(w_btn, "clicks", function(change) {
  ipywidgets::with_output(w_out, {
    tryCatch({
      suppressPackageStartupMessages({
        library(udpipe); library(writexl)
        library(dplyr); library(ggplot2); library(zip)
      })
      .prog("Loading texts...", 10)
      texts <- load_texts("notebooks/MyTexts")
      ok(paste("Loaded", length(texts), "file(s)"))
      .prog("Loading language model...", 25)
      model_dir <- file.path(Sys.getenv("HOME", "/home/jovyan"), "udpipe-models")
      dir.create(model_dir, showWarnings=FALSE, recursive=TRUE)
      existing <- list.files(model_dir,
        pattern=paste0("^", w_lang$value, ".*\\.udpipe$"),
        full.names=TRUE, recursive=TRUE)
      if (length(existing) > 0) {
        model_path <- existing[1]
        ok(paste("Using pre-loaded model:", basename(model_path)))
      } else {
        info(paste0("Downloading model (30-60 seconds)..."))
        dl <- udpipe_download_model(language=w_lang$value, model_dir=model_dir)
        if (isTRUE(dl$download_failed))
          stop("Model download failed. Check internet connection.")
        model_path <- dl$file_model
        ok(paste("Downloaded:", basename(model_path)))
      }
      model <- udpipe_load_model(model_path)
      .prog("Tagging texts...", 50)
      tagged_list <- lapply(seq_along(texts), function(i) {
        nm <- names(texts)[i]
        .prog(paste0("Tagging: ", nm, " (", i, "/", length(texts), ")"),
              50 + 35*(i/length(texts)))
        ann <- udpipe_annotate(model, x=texts[[nm]], doc_id=nm)
        as.data.frame(ann)
      })
      tagged_df <- dplyr::bind_rows(tagged_list) |>
        dplyr::select(doc_id, sentence_id, token_id, token, lemma,
                      upos, xpos, dep_rel, head_token_id) |>
        dplyr::rename(Document=doc_id, Sentence=sentence_id,
                      TokenID=token_id, Token=token, Lemma=lemma,
                      UPOS=upos, XPOS=xpos, DepRel=dep_rel,
                      HeadTokenID=head_token_id)
      .prog("Plotting POS summary...", 88)
      pos_counts <- tagged_df |>
        dplyr::filter(!is.na(UPOS), !UPOS %in% c("PUNCT","SPACE")) |>
        dplyr::count(UPOS, sort=TRUE)
      p <- ggplot(pos_counts, aes(x=reorder(UPOS,n), y=n)) +
        geom_col(fill="#51247a", width=.7) +
        geom_text(aes(label=format(n, big.mark=",")),
                  hjust=-0.1, size=3.5, colour="gray30") +
        coord_flip() +
        scale_y_continuous(expand=expansion(mult=c(0,.15))) +
        theme_bw(base_size=12) +
        labs(title="Token frequency by POS category",
             x="Universal POS tag", y="Count")
      print(p)
      .prog("Saving...", 95)
      save_excel(tagged_df, "pos_tagged.xlsx")
      ggsave("notebooks/MyOutput/pos_summary.png",
             bg="white", width=7, height=5, dpi=200)
      tmp <- tempfile(); dir.create(tmp)
      for (doc in unique(tagged_df$Document)) {
        df <- dplyr::filter(tagged_df, Document==doc,
                            !is.na(Token), !is.na(UPOS))
        writeLines(paste(paste0(df$Token,"_",df$UPOS), collapse=" "),
                   file.path(tmp, paste0(doc,"_postag.txt")))
      }
      zip::zip("notebooks/MyOutput/pos_tagged_texts.zip",
               files=list.files(tmp, full.names=TRUE), mode="cherry-pick")
      .prog("Done.", 100)
      ok(paste0("Tagged <b>", nrow(tagged_df), " tokens</b>. ",
                "Saved pos_tagged.xlsx, pos_summary.png, pos_tagged_texts.zip."))
    }, error=function(e) err(conditionMessage(e)))
  })
})


---

### Citation

Schweinberger, Martin. (2025). *LADAL Part-of-Speech Tagger*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025postagger,
  author = {Schweinberger, Martin},
  title  = {LADAL Part-of-Speech Tagger},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()